In [2]:

import h5py
import json
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    # bin 좌표 확인 (masks/filtered)
    row = f['masks/filtered/row'][:]
    col = f['masks/filtered/col'][:]
    print('bin 좌표 샘플 (row, col):', list(zip(row[:5], col[:5])))
    
    # feature 이름 확인 (유전자 이름)
    gene_names = f['features/name'][:]
    print('유전자 수:', len(gene_names))
    print('유전자 샘플:', gene_names[:5])
    
    # feature_slices 키 몇 개인지
    print('feature_slices 수 (유전자 수?):', len(f['feature_slices'].keys()))


bin 좌표 샘플 (row, col): [(0, 118), (0, 119), (0, 120), (0, 121), (0, 122)]
유전자 수: 37082
유전자 샘플: [b'MIR1302-2HG' b'FAM138A' b'OR4F5' b'AL627309.1' b'AL627309.3']
feature_slices 수 (유전자 수?): 18974


In [1]:

import h5py
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    # feature_slices의 key가 gene index인지 확인
    keys = list(f['feature_slices'].keys())
    print('첫 5개 key:', keys[:5])
    
    # 첫 번째 gene slice 확인
    first_key = keys[0]
    row  = f[f'feature_slices/{first_key}/row'][:]
    col  = f[f'feature_slices/{first_key}/col'][:]
    data = f[f'feature_slices/{first_key}/data'][:]
    print(f'gene index {first_key}: {len(data)}개 bin에서 발현')
    print(f'  row 샘플: {row[:5]}')
    print(f'  col 샘플: {col[:5]}')
    print(f'  count 샘플: {data[:5]}')
    
    # gene index → gene name 매핑 확인
    gene_names = f['features/name'][:]
    gene_ids   = f['features/id'][:]
    print(f'gene index {first_key} → name: {gene_names[int(first_key)]}')


첫 5개 key: ['1000', '10002', '10003', '10004', '10005']
gene index 1000: 14730개 bin에서 발현
  row 샘플: [0 0 1 1 1]
  col 샘플: [ 907 1809  802  866 1697]
  count 샘플: [1 1 1 1 1]
gene index 1000 → name: b'MAST2'


In [ ]:
근데 중요한 문제
bin 좌표가 **픽셀(px)이 아니라 bin grid 좌표 (0~3349)**예요.
타일은 픽셀 좌표로 자르니까 변환이 필요해요:
bin grid (0~3349)  →  이미지 픽셀 (0~5259/6000)

bin_px_x = bin_col × (img_width  / ncols)
         = bin_col × (5259 / 3350) ≈ bin_col × 1.570

bin_px_y = bin_row × (img_height / nrows)
         = bin_row × (6000 / 3350) ≈ bin_row × 1.791
이 변환으로 "이 타일 안에 어떤 bin이 속하는지" 찾을 수 있어요.

In [3]:
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 5.5 MB/s eta 0:00:0000:0100:01


In [4]:

import pandas as pd
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/spatial/tissue_positions.parquet', engine='fastparquet')
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print(df.head(3))


shape: (11222500, 6)
columns: ['barcode', 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres']
                 barcode  in_tissue  array_row  array_col  pxl_row_in_fullres  \
0  s_002um_00000_00000-1          1          0          0        31742.480385   
1  s_002um_00000_00001-1          1          0          1        31742.449772   
2  s_002um_00000_00002-1          1          0          2        31742.419159   

   pxl_col_in_fullres  
0        24365.998471  
1        24373.305191  
2        24380.611911  


In [5]:
# 저장된 타일 몇 개 샘플링해서 위치 확인

from PIL import Image
import numpy as np
import os

tile_dir = '/project_antwerp/hbae/data/visium_hd_tonsil/tiles_68um/'
tiles = sorted([f for f in os.listdir(tile_dir) if f.endswith('.png')])

print(f'총 타일 수: {len(tiles)}')

white_count = 0
for t in tiles[:100]:
    img = np.array(Image.open(f'{tile_dir}/{t}'))
    mean_brightness = img.mean()
    if mean_brightness > 200:
        white_count += 1
        print(f'white: {t}, brightness: {mean_brightness:.1f}')

print(f'흰색 타일: {white_count}/100')




#그리고 이 타일의 좌표가 `2166;5871`인데 hires 이미지 크기가 `5259×6000`이라서:

#col=2166 → 이미지 중간
#row=5871 → 이미지 아래쪽 (거의 끝)

총 타일 수: 7601
white: tissue_hires_image_1026;0.png, brightness: 238.3
white: tissue_hires_image_1026;1026.png, brightness: 237.5
white: tissue_hires_image_1026;1083.png, brightness: 237.3
white: tissue_hires_image_1026;114.png, brightness: 238.5
white: tissue_hires_image_1026;1140.png, brightness: 237.2
white: tissue_hires_image_1026;1197.png, brightness: 237.4
white: tissue_hires_image_1026;1254.png, brightness: 237.8
white: tissue_hires_image_1026;1311.png, brightness: 238.0
white: tissue_hires_image_1026;1368.png, brightness: 237.9
white: tissue_hires_image_1026;1425.png, brightness: 237.8
white: tissue_hires_image_1026;1482.png, brightness: 237.8
white: tissue_hires_image_1026;1539.png, brightness: 237.5
white: tissue_hires_image_1026;1596.png, brightness: 237.0
white: tissue_hires_image_1026;1653.png, brightness: 236.8
white: tissue_hires_image_1026;171.png, brightness: 238.8
white: tissue_hires_image_1026;1710.png, brightness: 236.5
white: tissue_hires_image_1026;1767.png, brightn

In [7]:
# 마스크 체크
import h5py, json
import numpy as np
import cv2
from PIL import Image

# 마스크 로드
with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    row  = f['masks/filtered/row'][:]
    col  = f['masks/filtered/col'][:]
    meta = json.loads(f.attrs['metadata_json'])

nrows, ncols = meta['nrows'], meta['ncols']
mask = np.zeros((nrows, ncols), dtype=np.uint8)
mask[row, col] = 1

# 이미지 크기로 리사이즈
img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png'))
img_h, img_w = img.shape[:2]
mask_resized = cv2.resize(mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)

# 문제 타일 확인 (col=1026, row=0)
col_start, row_start = 1026, 0
tile_size = 57
mask_tile = mask_resized[row_start:row_start+tile_size, col_start:col_start+tile_size]
print(f'타일 (col=1026, row=0) 마스크 비율: {mask_tile.mean():.3f}')
print(f'마스크 tile 샘플:\n{mask_tile[:5,:5]}')

# 마스크 전체 시각화 저장
mask_vis = (mask_resized * 255).astype(np.uint8)
Image.fromarray(mask_vis).save('/project_antwerp/hbae/data/visium_hd_tonsil/mask_debug.png')
print(f'마스크 저장: /tmp/mask_debug.png')
print(f'마스크 조직 비율: {mask_resized.mean():.3f}')


타일 (col=1026, row=0) 마스크 비율: 1.000
마스크 tile 샘플:
[[1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]
 [1 1 1 1 1]]
마스크 저장: /tmp/mask_debug.png
마스크 조직 비율: 0.785


In [9]:
# 이미지와 마스크 비교

import h5py, json
import numpy as np
import cv2
from PIL import Image
import pandas as pd

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    row  = f['masks/filtered/row'][:]
    col  = f['masks/filtered/col'][:]
    meta = json.loads(f.attrs['metadata_json'])

nrows, ncols = meta['nrows'], meta['ncols']
mask = np.zeros((nrows, ncols), dtype=np.uint8)
mask[row, col] = 1

img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png').convert('RGB'))
img_h, img_w = img.shape[:2]
mask_resized = cv2.resize(mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)

# 마스크 overlay 저장
overlay = img.copy()
overlay[mask_resized == 0] = [255, 0, 0]
Image.fromarray(overlay).save('/project_antwerp/hbae/data/visium_hd_tonsil/mask_overlay.png')
print('overlay 저장: /project_antwerp/hbae/data/visium_hd_tonsil/mask_overlay.png')

# pyarrow로 변경
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/spatial/tissue_positions.parquet', engine='pyarrow')
print(f'in_tissue=1: {df.in_tissue.sum():,} / 전체: {len(df):,}')
print(f'in_tissue 비율: {df.in_tissue.mean():.3f}')
print(df.head(3))


overlay 저장: /project_antwerp/hbae/data/visium_hd_tonsil/mask_overlay.png
in_tissue=1: 11,210,791 / 전체: 11,222,500
in_tissue 비율: 0.999
                 barcode  in_tissue  array_row  array_col  pxl_row_in_fullres  \
0  s_002um_00000_00000-1          1          0          0        31742.480385   
1  s_002um_00000_00001-1          1          0          1        31742.449772   
2  s_002um_00000_00002-1          1          0          2        31742.419159   

   pxl_col_in_fullres  
0        24365.998471  
1        24373.305191  
2        24380.611911  


In [10]:

import pandas as pd
import numpy as np
import json

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/spatial/tissue_positions.parquet', engine='pyarrow')
scalef = 0.09202595

# fullres → hires 변환
df['hires_row'] = df['pxl_row_in_fullres'] * scalef
df['hires_col'] = df['pxl_col_in_fullres'] * scalef

print('hires_row 범위:', df['hires_row'].min(), '~', df['hires_row'].max())
print('hires_col 범위:', df['hires_col'].min(), '~', df['hires_col'].max())
print()

# 이미지 크기는 5259×6000인데 범위 밖인 bin 확인
out_of_range = df[(df['hires_row'] < 0) | (df['hires_row'] > 6000) | 
                  (df['hires_col'] < 0) | (df['hires_col'] > 5259)]
print(f'범위 밖 bin: {len(out_of_range):,}')


hires_row 범위: 660.11468726171 ~ 2921.131912785103
hires_col 범위: 2232.649660441529 ~ 4494.186987967945

범위 밖 bin: 0


In [11]:
# 틀린거같음
import pandas as pd
import numpy as np
import cv2
from PIL import Image

scalef = 0.09202595
img_w, img_h = 5259, 6000

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/spatial/tissue_positions.parquet', engine='pyarrow')

# hires 좌표로 변환
df['hires_row'] = (df['pxl_row_in_fullres'] * scalef).astype(int)
df['hires_col'] = (df['pxl_col_in_fullres'] * scalef).astype(int)

# 마스크 생성
mask = np.zeros((img_h, img_w), dtype=np.uint8)
mask[df['hires_row'].values, df['hires_col'].values] = 1

print(f'마스크 조직 비율: {mask.mean():.4f}')
print(f'조직 픽셀 수: {mask.sum():,}')

# 마스크 시각화
img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png').convert('RGB'))
overlay = img.copy()
overlay[mask == 0] = [255, 0, 0]
Image.fromarray(overlay).save('/project_antwerp/hbae/data/visium_hd_tonsil/mask_overlay_parquet.png')
print('overlay 저장 완료')


마스크 조직 비율: 0.1608
조직 픽셀 수: 5,074,844
overlay 저장 완료


In [12]:

import h5py, json
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta = json.loads(f.attrs['metadata_json'])
    print('transform_matrices:')
    matrices = json.loads(meta['hd_layout_json']) if 'hd_layout_json' in meta else meta.get('transform_matrices', {})
    print(json.dumps(meta.get('transform_matrices', {}), indent=2))


transform_matrices:
{
  "spot_colrow_to_microscope_colrow": [
    [
      -3.65394524181981,
      0.002133389590472481,
      15524.64179357808
    ],
    [
      0.0030534266764736047,
      3.6526930255968555,
      2020.7158251302426
    ],
    [
      1.8743537078861507e-08,
      -4.7170217319431605e-09,
      1.0
    ]
  ],
  "microscope_colrow_to_spot_colrow": [
    [
      -0.2736773693265364,
      0.0001653301507703104,
      4248.409040551218
    ],
    [
      0.00022593905138911606,
      0.27379227906318326,
      -556.7640139414468
    ],
    [
      5.130747679032707e-09,
      1.2883852585680649e-09,
      1.0
    ]
  ],
  "spot_colrow_to_cytassist_colrow": [
    [
      0.4251760225321907,
      0.0006365874222225936,
      473.9569058878465
    ],
    [
      -0.0005820247213734044,
      0.42592803749117775,
      768.9548981287641
    ],
    [
      -2.2273249612892457e-07,
      2.3497890391959577e-07,
      1.0
    ]
  ],
  "cytassist_colrow_to_spot_colrow": [
 

In [13]:

import h5py, json
import numpy as np
import cv2
from PIL import Image

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta   = json.loads(f.attrs['metadata_json'])
    row    = f['masks/filtered/row'][:]
    col    = f['masks/filtered/col'][:]

# 변환 행렬 (spot_colrow → microscope_colrow, fullres 기준)
M = np.array(meta['transform_matrices']['spot_colrow_to_microscope_colrow'])

# bin grid 좌표 (col, row) → homogeneous
nrows, ncols = meta['nrows'], meta['ncols']
pts = np.stack([col, row, np.ones(len(row))], axis=1).T  # (3, N)

# fullres 픽셀 좌표로 변환
px = M @ pts  # (3, N)
px_col = px[0] / px[2]  # fullres col
px_row = px[1] / px[2]  # fullres row

# hires로 변환
scalef = 0.09202595
hires_col = (px_col * scalef).astype(int)
hires_row = (px_row * scalef).astype(int)

img_w, img_h = 5259, 6000
valid = (hires_col >= 0) & (hires_col < img_w) & (hires_row >= 0) & (hires_row < img_h)
print(f'유효 bin: {valid.sum():,} / 전체: {len(valid):,}')

# 마스크 생성
mask = np.zeros((img_h, img_w), dtype=np.uint8)
mask[hires_row[valid], hires_col[valid]] = 1

# dilate
kernel = np.ones((15,15), np.uint8)
mask_dilated = cv2.dilate(mask, kernel, iterations=1)
print(f'조직 비율: {mask_dilated.mean():.4f}')

# 시각화
img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png').convert('RGB'))
overlay = img.copy()
overlay[mask_dilated == 0] = [255, 0, 0]
Image.fromarray(overlay).save('/project_antwerp/hbae/data/visium_hd_tonsil/mask_overlay_transform.png')
print('저장 완료')


유효 bin: 8,812,974 / 전체: 8,812,974
조직 비율: 0.0339
저장 완료


In [14]:

import h5py, json
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta = json.loads(f.attrs['metadata_json'])
    row  = f['masks/filtered/row'][:]
    col  = f['masks/filtered/col'][:]

M = np.array(meta['transform_matrices']['spot_colrow_to_microscope_colrow'])
pts = np.stack([col, row, np.ones(len(row))], axis=1).T

px = M @ pts
px_col = px[0] / px[2]
px_row = px[1] / px[2]

print('fullres col 범위:', px_col.min(), '~', px_col.max())
print('fullres row 범위:', px_row.min(), '~', px_row.max())

scalef = 0.09202595
print('hires col 범위:', (px_col * scalef).min(), '~', (px_col * scalef).max())
print('hires row 범위:', (px_row * scalef).min(), '~', (px_row * scalef).max())

# fullres 이미지 크기 추정
print()
print('hires 이미지 크기: 5259 x 6000')
print('fullres 이미지 크기 추정:')
print('  width:', 5259 / scalef)
print('  height:', 6000 / scalef)


fullres col 범위: 3289.742919472729 ~ 15531.524341566315
fullres row 범위: 2021.0716593983393 ~ 14262.31603862526
hires col 범위: 302.7417174202514 ~ 1429.3032824807647
hires row 범위: 185.9910394742086 ~ 1312.5031826547263

hires 이미지 크기: 5259 x 6000
fullres 이미지 크기 추정:
  width: 57146.924318629695
  height: 65199.000933975694


In [15]:

from PIL import Image
import numpy as np

# 각 이미지 크기 확인
imgs = {
    'tissue_hires': '/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png',
    'aligned_tissue': '/project_antwerp/hbae/data/visium_hd_tonsil/spatial/aligned_tissue_image.jpg',
    'cytassist': '/project_antwerp/hbae/data/visium_hd_tonsil/spatial/cytassist_image.tiff',
}
for name, path in imgs.items():
    img = Image.open(path)
    print(f'{name}: {img.size} ({img.mode})')


tissue_hires: (5259, 6000) (RGB)
aligned_tissue: (5259, 6000) (RGB)
cytassist: (3200, 3000) (RGB)


In [16]:

import h5py, json
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta = json.loads(f.attrs['metadata_json'])

# scalefactors 확인
import json as j
sf = j.load(open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/spatial/scalefactors_json.json'))
print('scalefactors:')
for k,v in sf.items():
    print(f'  {k}: {v}')

print()
print('microns_per_pixel:', sf['microns_per_pixel'])
print('tissue_hires_scalef:', sf['tissue_hires_scalef'])
print()

# fullres 크기 추정
# fullres col 범위: 3289~15531
# tissue_hires_scalef로 나누면 hires 범위
scalef = sf['tissue_hires_scalef']
print(f'fullres col 3289~15531 → hires: {3289*scalef:.0f}~{15531*scalef:.0f}')
print(f'fullres row 2021~14262 → hires: {2021*scalef:.0f}~{14262*scalef:.0f}')
print()
print(f'hires 이미지 크기: 5259×6000')
print(f'예상 fullres 크기: {5259/scalef:.0f}×{6000/scalef:.0f}')


scalefactors:
  spot_diameter_fullres: 7.306325414203511
  bin_size_um: 2.0
  microns_per_pixel: 0.273735412347224
  tissue_lowres_scalef: 0.009202595
  fiducial_diameter_fullres: 1205.5436933435794
  tissue_hires_scalef: 0.09202595
  regist_target_img_scalef: 0.09202595

microns_per_pixel: 0.273735412347224
tissue_hires_scalef: 0.09202595

fullres col 3289~15531 → hires: 303~1429
fullres row 2021~14262 → hires: 186~1312

hires 이미지 크기: 5259×6000
예상 fullres 크기: 57147×65199


In [18]:

import pandas as pd
import numpy as np
import cv2
from PIL import Image

sf = 0.09202595
img_w, img_h = 5259, 6000

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/spatial/tissue_positions.parquet', engine='pyarrow')
df['hires_row'] = (df['pxl_row_in_fullres'] * sf).astype(int)
df['hires_col'] = (df['pxl_col_in_fullres'] * sf).astype(int)
df = df[(df['hires_row'] >= 0) & (df['hires_row'] < img_h) &
        (df['hires_col'] >= 0) & (df['hires_col'] < img_w)]

mask = np.zeros((img_h, img_w), dtype=np.uint8)
mask[df['hires_row'].values, df['hires_col'].values] = 1

# tile_size(57px) 기준으로 dilate
# 57×57 타일 안에 bin이 있으면 조직으로 처리
kernel = np.ones((57, 57), np.uint8)
mask_dilated = cv2.dilate(mask, kernel, iterations=1)
print(f'조직 비율: {mask_dilated.mean():.4f}')

img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png').convert('RGB'))
overlay = img.copy()
overlay[mask_dilated == 0] = [255, 0, 0]
Image.fromarray(overlay).save('/project_antwerp/hbae/data/visium_hd_tonsil/mask_overlay_v4.png')
np.save('/project_antwerp/hbae/data/visium_hd_tonsil/tissue_mask_hires.npy', mask_dilated)
print('완료')


조직 비율: 0.1690
완료


In [19]:
# detected_tissue_image.jpg 파일이 있었잖아요. 이게 Space Ranger가 자동으로 감지한 조직 마스크예요:
from PIL import Image
import numpy as np

# detected_tissue_image 확인
img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/detected_tissue_image.jpg'))
print('detected_tissue_image size:', img.shape)
print('unique values:', np.unique(img))
print('mean:', img.mean())


detected_tissue_image size: (3000, 3200, 3)
unique values: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 

In [21]:
# 그리고 masks/filtered가 어떤 좌표계인지 다시 확인해볼게요:

import h5py, json
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta = json.loads(f.attrs['metadata_json'])
    row  = f['masks/filtered/row'][:]
    col  = f['masks/filtered/col'][:]

nrows = meta['nrows']
ncols = meta['ncols']
print('masks/filtered row 범위:', row.min(), '~', row.max())
print('masks/filtered col 범위:', col.min(), '~', col.max())
print('nrows:', nrows, 'ncols:', ncols)
print()
print('row.max() vs nrows:', row.max(), 'vs', nrows)
print('col.max() vs ncols:', col.max(), 'vs', ncols)

# detected_tissue_image 확인
from PIL import Image
img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/detected_tissue_image.jpg'))
print()
print('detected_tissue_image shape:', img.shape)
print('mean brightness:', img.mean())



masks/filtered row 범위: 0 ~ 3349
masks/filtered col 범위: 0 ~ 3349
nrows: 3350 ncols: 3350

row.max() vs nrows: 3349 vs 3350
col.max() vs ncols: 3349 vs 3350

detected_tissue_image shape: (3000, 3200, 3)
mean brightness: 189.4090601388889


In [ ]:
(base) hyobaeug@3f0333da8f5a:/project_antwerp/hbae/data/visium_hd_tonsil/spatial$ ls

aligned_fiducials.jpg  aligned_tissue_image.jpg  cytassist_image.tiff  detected_tissue_image.jpg  tissue_hires_image.png  tissue_lowres_image.png



이미지 종류가 이렇게 있어

In [22]:

import h5py, json
import numpy as np

with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    print('masks 하위 키들:')
    for key in f['masks'].keys():
        row  = f[f'masks/{key}/row'][:]
        col  = f[f'masks/{key}/col'][:]
        data = f[f'masks/{key}/data'][:]
        print(f'  masks/{key}: {len(data)}개 entries, row {row.min()}~{row.max()}, col {col.min()}~{col.max()}, data {data.min()}~{data.max()}')
    
    print()
    print('images 하위 키들:')
    for key in f['images'].keys():
        d = f[f'images/{key}/data'][:]
        r = f[f'images/{key}/row'][:]
        c = f[f'images/{key}/col'][:]
        attrs = dict(f[f'images/{key}'].attrs)
        print(f'  images/{key}: data shape {d.shape}, row {r.min()}~{r.max()}, col {c.min()}~{c.max()}')
        print(f'    attrs: {attrs}')


masks 하위 키들:
  masks/filtered: 8812974개 entries, row 0~3349, col 0~3349, data 1~1
  masks/square_008um: 553820개 entries, row 0~837, col 0~837, data 1~1
  masks/square_016um: 139406개 entries, row 0~418, col 0~418, data 1~1

images 하위 키들:
  images/cytassist: data shape (11222499,), row 0~3349, col 0~3349
    attrs: {'metadata_json': '{"kind": "gray_image_u8", "domain": "all_spots", "binning_scale": 1}'}
  images/microscope: data shape (10998169,), row 0~3349, col 62~3349
    attrs: {'metadata_json': '{"kind": "gray_image_u8", "domain": "all_spots", "binning_scale": 1}'}


In [23]:
# Otsu thresholding이 가장 확실해요. 아까 코드 바로 실행해볼게요:
import numpy as np
from PIL import Image
from skimage.color import rgb2hsv
from skimage.filters import threshold_otsu

img_RGB = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png').convert('RGB'))

img_HSV = rgb2hsv(img_RGB)
background_R = img_RGB[:,:,0] > threshold_otsu(img_RGB[:,:,0])
background_G = img_RGB[:,:,1] > threshold_otsu(img_RGB[:,:,1])
background_B = img_RGB[:,:,2] > threshold_otsu(img_RGB[:,:,2])
tissue_RGB = np.logical_not(background_R & background_G & background_B)
tissue_S = img_HSV[:,:,1] > threshold_otsu(img_HSV[:,:,1])
min_R = img_RGB[:,:,0] > 50
min_G = img_RGB[:,:,1] > 50
min_B = img_RGB[:,:,2] > 50
mask = (tissue_S & tissue_RGB & min_R & min_G & min_B).astype(np.uint8)

print('조직 비율:', mask.mean())

overlay = img_RGB.copy()
overlay[mask == 0] = [255, 0, 0]
Image.fromarray(overlay).save('/project_antwerp/hbae/data/visium_hd_tonsil/mask_otsu.png')
np.save('/project_antwerp/hbae/data/visium_hd_tonsil/tissue_mask_otsu.npy', mask)
print('저장 완료')


조직 비율: 0.2693423654687203
저장 완료


In [24]:

import numpy as np
from PIL import Image
import cv2

img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png').convert('RGB'))

# 밝기 기반: 밝은 픽셀(배경)만 제거
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
mask = (gray < 220).astype(np.uint8)

# morphological closing으로 구멍 메우기
kernel = np.ones((20,20), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

print('조직 비율:', mask.mean())
overlay = img.copy()
overlay[mask == 0] = [255, 0, 0]
Image.fromarray(overlay).save('/project_antwerp/hbae/data/visium_hd_tonsil/mask_brightness.png')
np.save('/project_antwerp/hbae/data/visium_hd_tonsil/tissue_mask_brightness.npy', mask)
print('저장 완료')


조직 비율: 0.5776910058946568
저장 완료
